# Step 9 — Verification and conservation

**Learning objective:** Check OGS boundary nodal values, discrete FEM conservation, and 2/3 flux scaling.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Run:** validated numerical tests. Note that these are discrete FEM boundary reactions in model units, not physical m³/s.

In [ ]:
run(sys.executable,'scripts/check_results.py')
run(sys.executable,'scripts/check_flux_balance.py')

**Inspect:** bar plot of source/sink reactions for both cases.

In [ ]:
import csv
with (ROOT/'results'/'discrete_flux_balance.csv').open() as f:
    rows=list(csv.DictReader(f))
names=[r['case'] for r in rows]
left=[float(r['left_injection']) for r in rows]
right=[float(r['right_injection']) for r in rows]
x=np.arange(len(names));width=.33
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(x-width/2,left,width,label='Left boundary reaction',color='#b91c1c')
ax.bar(x+width/2,right,width,label='Right boundary reaction',color='#1d4ed8')
ax.axhline(0,color='black',lw=.8)
ax.set_xticks(x,names);ax.set_ylabel('Discrete FEM reaction [model units]')
ax.set_title('Conservation: equal and opposite reactions');ax.legend();ax.grid(axis='y',alpha=.2)
plt.show()
for r in rows:
    print(r['case'], 'relative imbalance =', r['relative_imbalance'])
print('Flux amplitude ratio BC50/BC60:',left[1]/left[0])

**Interpretation / PASS:** Left and right bars show **signed, integrated FEM boundary reactions**: equal magnitude with opposite sign indicates discrete conservation. They are not physical volumetric groundwater flow rates. The BC50/BC60 amplitude ratio must be `2/3`. `check_flux_balance.py` independently assembles triangular-element reactions to make that check. **Final question:** Why is changing boundary values not equivalent to changing hydraulic conductivity?